# 06. NGBoost 결과 점검

**목적**: `pv/experiments/ngboost_baseline/test_predictions.parquet`에 저장된 2025 테스트셋 예측을 읽어,
- 사이트별 평균 편향
- 시간축 예측 vs 실제
- 불확실성(`pred_std_cf`) 분포
- 불확실성과 실제 오차의 관계

를 빠르게 확인한다.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

plt.rcParams["font.family"] = ["Malgun Gothic", "AppleGothic", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
sns.set_theme(style="whitegrid")

ROOT = Path("../../")
PRED_PATH = ROOT / "pv" / "experiments" / "ngboost_baseline" / "test_predictions.parquet"

df = pd.read_parquet(PRED_PATH)
df["datetime_kst"] = pd.to_datetime(df["datetime_kst"])
df["abs_err_cf"] = (df["cf"] - df["pred_cf"]).abs()
df["err_cf"] = df["pred_cf"] - df["cf"]
df["abs_err_kwh"] = df["abs_err_cf"] * df["site_capacity_kw"]
df["month"] = df["datetime_kst"].dt.month
df["date"] = df["datetime_kst"].dt.date

print("PATH:", PRED_PATH)
print("rows:", len(df))
print("period:", df["datetime_kst"].min(), "->", df["datetime_kst"].max())
print("sites:", sorted(df["site"].unique()))

## 1. 전체 요약

In [ ]:
summary = pd.DataFrame({
    "metric": ["rows", "actual_cf_mean", "pred_cf_mean", "pred_std_mean", "mae_cf"],
    "value": [
        len(df),
        df["cf"].mean(),
        df["pred_cf"].mean(),
        df["pred_std_cf"].mean(),
        df["abs_err_cf"].mean(),
    ],
})
summary

## 2. 사이트별 평균 성능

In [ ]:
site_summary = (
    df.groupby("site", as_index=False)
      .agg(
          rows=("site", "size"),
          actual_cf_mean=("cf", "mean"),
          pred_cf_mean=("pred_cf", "mean"),
          pred_std_mean=("pred_std_cf", "mean"),
          mae_cf=("abs_err_cf", "mean"),
          bias_cf=("err_cf", "mean"),
      )
      .sort_values("actual_cf_mean", ascending=False)
)
site_summary.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(data=site_summary, x="site", y="actual_cf_mean", color="#4C78A8", ax=axes[0])
axes[0].plot(site_summary["site"], site_summary["pred_cf_mean"], color="#E45756", marker="o", linewidth=2)
axes[0].set_title("사이트별 평균 CF: 실제 vs 예측")
axes[0].set_ylabel("capacity factor")
axes[0].tick_params(axis="x", rotation=30)

sns.barplot(data=site_summary, x="site", y="bias_cf", color="#72B7B2", ax=axes[1])
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("사이트별 평균 bias (pred - actual)")
axes[1].set_ylabel("bias in CF")
axes[1].tick_params(axis="x", rotation=30)

sns.barplot(data=site_summary, x="site", y="pred_std_mean", color="#F58518", ax=axes[2])
axes[2].set_title("사이트별 평균 예측 표준편차")
axes[2].set_ylabel("mean predicted std")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

→ **판독 포인트**:
- `actual_cf_mean`과 `pred_cf_mean` 차이가 크면 사이트별 평균 편향 존재
- `pred_std_mean`이 큰 사이트는 모델이 그 사이트를 더 불확실하게 본 것
- `bias_cf` 부호가 일정하면 site-specific calibration 필요 가능

## 3. 시간축 예시: 사이트별 일평균 CF

In [ ]:
daily = (
    df.groupby(["site", "date"], as_index=False)
      .agg(actual_cf=("cf", "mean"), pred_cf=("pred_cf", "mean"), pred_std=("pred_std_cf", "mean"))
)

sites = sorted(daily["site"].unique())
fig, axes = plt.subplots(4, 2, figsize=(16, 12), sharex=True, sharey=True)
axes = axes.ravel()

for ax, site in zip(axes, sites):
    g = daily[daily["site"] == site]
    ax.plot(g["date"], g["actual_cf"], label="actual", color="#4C78A8", linewidth=1.5)
    ax.plot(g["date"], g["pred_cf"], label="pred", color="#E45756", linewidth=1.2)
    ax.set_title(site)
    ax.tick_params(axis="x", rotation=30)

for ax in axes[len(sites):]:
    ax.axis("off")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2)
fig.suptitle("2025 사이트별 일평균 CF: 실제 vs 예측", y=1.02)
plt.tight_layout()
plt.show()

## 4. 포트폴리오 합산 시계열

In [ ]:
portfolio = (
    df.assign(actual_kwh=lambda x: x["cf"] * x["site_capacity_kw"], pred_kwh=lambda x: x["pred_cf"] * x["site_capacity_kw"])
      .groupby("datetime_kst", as_index=False)
      .agg(actual_kwh=("actual_kwh", "sum"), pred_kwh=("pred_kwh", "sum"))
)
portfolio["date"] = portfolio["datetime_kst"].dt.date
portfolio_daily = portfolio.groupby("date", as_index=False).agg(actual_kwh=("actual_kwh", "sum"), pred_kwh=("pred_kwh", "sum"))

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(portfolio_daily["date"], portfolio_daily["actual_kwh"], label="actual", color="#4C78A8")
ax.plot(portfolio_daily["date"], portfolio_daily["pred_kwh"], label="pred", color="#E45756")
ax.set_title("포트폴리오 일합계 발전량: 실제 vs 예측")
ax.set_ylabel("kWh/day")
ax.tick_params(axis="x", rotation=30)
ax.legend()
plt.tight_layout()
plt.show()

## 5. 예측 불확실성 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["pred_std_cf"], bins=50, kde=True, color="#F58518", ax=axes[0])
axes[0].set_title("전체 pred_std_cf 분포")
axes[0].set_xlabel("predicted std of CF")

sns.boxplot(data=df, x="site", y="pred_std_cf", color="#72B7B2", ax=axes[1], showfliers=False)
axes[1].set_title("사이트별 pred_std_cf 분포")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 6. 불확실성과 실제 오차의 관계

In [ ]:
corr = df[["pred_std_cf", "abs_err_cf"]].corr().iloc[0, 1]
print(f"corr(pred_std_cf, abs_err_cf) = {corr:.3f}")

std_bins = pd.qcut(df["pred_std_cf"], q=10, duplicates="drop")
bin_summary = (
    df.assign(std_bin=std_bins)
      .groupby("std_bin", as_index=False)
      .agg(pred_std_mean=("pred_std_cf", "mean"), abs_err_mean=("abs_err_cf", "mean"), rows=("site", "size"))
)
bin_summary.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample = df.sample(min(5000, len(df)), random_state=42)
sns.scatterplot(data=sample, x="pred_std_cf", y="abs_err_cf", s=15, alpha=0.35, ax=axes[0])
axes[0].set_title("예측 std vs 실제 절대오차")

axes[1].plot(bin_summary["pred_std_mean"], bin_summary["abs_err_mean"], marker="o", color="#E45756")
axes[1].set_title("pred_std decile별 평균 오차")
axes[1].set_xlabel("mean predicted std")
axes[1].set_ylabel("mean absolute error")

plt.tight_layout()
plt.show()

→ **판독 포인트**:
- `pred_std_cf`가 큰 구간에서 `abs_err_cf`도 같이 커지면 uncertainty head가 의미 있게 작동한 것
- 상관이 거의 없으면 NGBoost 분포 출력이 calibration 측면에서 약할 수 있음

## 7. 80% / 95% 구간 coverage

In [ ]:
z80 = 1.28
z95 = 1.96

df["lo80"] = df["pred_cf"] - z80 * df["pred_std_cf"]
df["hi80"] = df["pred_cf"] + z80 * df["pred_std_cf"]
df["lo95"] = df["pred_cf"] - z95 * df["pred_std_cf"]
df["hi95"] = df["pred_cf"] + z95 * df["pred_std_cf"]

overall = pd.DataFrame({
    "interval": ["80%", "95%"],
    "coverage": [
        ((df["cf"] >= df["lo80"]) & (df["cf"] <= df["hi80"])).mean(),
        ((df["cf"] >= df["lo95"]) & (df["cf"] <= df["hi95"])).mean(),
    ],
})
overall

In [ ]:
coverage_site = (
    df.groupby("site", as_index=False)
      .agg(
          cov80=("cf", lambda s: ((s >= df.loc[s.index, "lo80"]) & (s <= df.loc[s.index, "hi80"])).mean()),
          cov95=("cf", lambda s: ((s >= df.loc[s.index, "lo95"]) & (s <= df.loc[s.index, "hi95"])).mean()),
      )
)
coverage_site.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(coverage_site))
w = 0.35
ax.bar(x - w/2, coverage_site["cov80"], width=w, label="80% interval", color="#4C78A8")
ax.bar(x + w/2, coverage_site["cov95"], width=w, label="95% interval", color="#F58518")
ax.axhline(0.80, color="#4C78A8", linestyle="--", linewidth=1)
ax.axhline(0.95, color="#F58518", linestyle="--", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(coverage_site["site"], rotation=30)
ax.set_ylim(0, 1.05)
ax.set_title("사이트별 예측구간 coverage")
ax.legend()
plt.tight_layout()
plt.show()